# 🏥 Diabetic Retinopathy Grading Pipeline (Production-Ready)
### Optimized for NVIDIA RTX 2050 (4GB VRAM) | 16GB RAM | i5-12450H

**30-Step Complete Pipeline** — Every cell runs fresh, no skipping.

| Spec | Value |
|------|-------|
| GPU | NVIDIA GeForce RTX 2050 (4 GB) |
| CPU | 12th Gen Intel i5-12450H |
| RAM | 16 GB |
| Storage | ~247 GB free |
| Strategy | Mixed Precision (FP16) + Gradient Accumulation |

> ⚠️ **Run cells in order. Each cell is self-contained and will NOT skip.**


## 📦 Step 1 — System Setup & Hardware Detection
Checks GPU, CUDA, memory, and configures device.

In [ ]:
# — Step 1: System Setup & Hardware Detection —
import os, sys, platform, subprocess, shutil, time, warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("  SYSTEM INFORMATION")
print("=" * 60)
print(f"  OS: {platform.system()} {platform.release()}")
print(f"  Python: {sys.version.split()[0]}")
print(f"  CPU: {platform.processor()}")

# Memory check
import psutil
ram = psutil.virtual_memory()
print(f"  RAM: {ram.total / 1e9:.1f} GB (Available: {ram.available / 1e9:.1f} GB)")

# Storage check
disk = shutil.disk_usage('.')
print(f"  Storage: {disk.free / 1e9:.1f} GB free of {disk.total / 1e9:.1f} GB")

# GPU check
import torch
print(f"\n  PyTorch: {torch.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"  GPU: {gpu_name} ({gpu_mem:.1f} GB)")
    print(f"  CUDA version: {torch.version.cuda}")
    # Enable TF32 for Ampere+ GPUs
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
    print("  Using Apple MPS")
else:
    DEVICE = torch.device('cpu')
    print("  ⚠️ No GPU detected — using CPU (will be very slow)")

torch.backends.cudnn.benchmark = True
print(f"\n✅ Device selected: {DEVICE}")
print("=" * 60)


## 📥 Step 2 — Install Requirements
Installs all necessary packages.

In [ ]:
# — Step 2: Install Requirements —
import subprocess, sys

packages = [
    'timm>=0.9.0',
    'albumentations>=1.3.0',
    'opencv-python-headless>=4.8.0',
    'scikit-learn>=1.3.0',
    'scipy>=1.11.0',
    'tqdm',
    'pandas',
    'matplotlib',
    'seaborn',
    'Pillow',
    'opendatasets',
    'kaggle',
    'grad-cam',
    'streamlit',
]

print("Installing packages...")
for pkg in packages:
    name = pkg.split('>=')[0].split('==')[0]
    try:
        __import__(name.replace('-', '_'))
        print(f"  ✅ {name} already installed")
    except ImportError:
        print(f"  📦 Installing {pkg}...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
        print(f"  ✅ {name} installed")

print("\n✅ All requirements satisfied")


## 🔑 Step 3 — Kaggle Authentication
Upload your `kaggle.json` or set credentials manually.

**Option A:** Place `kaggle.json` in `~/.kaggle/kaggle.json`
**Option B:** Set the variables below manually.

In [ ]:
# — Step 3: Kaggle Authentication —
import os
from pathlib import Path

kaggle_dir = Path.home() / '.kaggle'
kaggle_json = kaggle_dir / 'kaggle.json'

if kaggle_json.exists():
    print(f"✅ Kaggle credentials found at {kaggle_json}")
else:
    # Option B: Set manually
    # KAGGLE_USERNAME = "your_username"
    # KAGGLE_KEY = "your_api_key"
    # kaggle_dir.mkdir(exist_ok=True)
    # with open(kaggle_json, 'w') as f:
    #     import json; json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
    # os.chmod(kaggle_json, 0o600)

    # Try upload widget (Jupyter)
    try:
        from google.colab import files
        print("Upload your kaggle.json:")
        uploaded = files.upload()
        kaggle_dir.mkdir(exist_ok=True)
        for fn, content in uploaded.items():
            with open(kaggle_json, 'wb') as f:
                f.write(content)
        os.chmod(kaggle_json, 0o600)
        print("✅ Kaggle credentials uploaded")
    except ImportError:
        if not kaggle_json.exists():
            print("⚠️ Please place kaggle.json in ~/.kaggle/ before proceeding")
            print("   Download from: https://www.kaggle.com/settings → API → Create New Token")

# Verify
try:
    import kaggle
    kaggle.api.authenticate()
    print("✅ Kaggle API authenticated successfully")
except Exception as e:
    print(f"⚠️ Kaggle auth issue: {e}")
    print("   You can still proceed if you download the dataset manually.")


## 📂 Step 4 — Dataset Download & Extraction (APTOS 2019)
Downloads the APTOS 2019 Blindness Detection dataset from Kaggle.

In [ ]:
# — Step 4: Dataset Download & Extraction —
import os, zipfile, time
from pathlib import Path

# Configuration
DATASET = "mariaherrerai/aptos2019"
DATA_DIR = Path("./data/aptos2019")
DATA_DIR.mkdir(parents=True, exist_ok=True)

train_csv = DATA_DIR / "train.csv"
train_images = DATA_DIR / "train_images"

if train_csv.exists() and train_images.exists() and len(list(train_images.glob("*.png"))) > 100:
    n_images = len(list(train_images.glob("*.png")))
    print(f"✅ Dataset already exists: {n_images} images found")
else:
    print("📥 Downloading APTOS 2019 dataset from Kaggle...")
    t0 = time.time()
    try:
        import kaggle
        kaggle.api.dataset_download_files(DATASET, path=str(DATA_DIR), unzip=True)
        print(f"✅ Download complete in {time.time()-t0:.0f}s")
    except Exception as e:
        print(f"⚠️ Kaggle download failed: {e}")
        print("\nManual download instructions:")
        print("1. Go to https://www.kaggle.com/datasets/mariaherrerai/aptos2019")
        print("2. Download and extract to ./data/aptos2019/")
        print("   Ensure train.csv and train_images/ folder exist")

    # Handle nested folders from extraction
    for sub in DATA_DIR.rglob("train.csv"):
        if sub.parent != DATA_DIR:
            import shutil
            for item in sub.parent.iterdir():
                dest = DATA_DIR / item.name
                if not dest.exists():
                    shutil.move(str(item), str(dest))
            print("  Restructured extracted files")
            break

# Verify
if train_csv.exists():
    import pandas as pd
    df_check = pd.read_csv(train_csv)
    print(f"✅ train.csv: {len(df_check)} entries")
if train_images.exists():
    n_imgs = len(list(train_images.glob("*.png")))
    print(f"✅ train_images/: {n_imgs} images")


## 📋 Step 5 — Load Dataset
Loads CSV, maps labels, constructs image paths.

In [ ]:
# — Step 5: Load Dataset —
import pandas as pd
from pathlib import Path

DATA_DIR = Path("./data/aptos2019")
TRAIN_CSV = DATA_DIR / "train.csv"
TRAIN_IMAGES = DATA_DIR / "train_images"

# Load CSV
df = pd.read_csv(TRAIN_CSV)
print(f"Raw CSV: {len(df)} entries")
print(f"Columns: {list(df.columns)}")

# Standardize column names
if 'id_code' in df.columns:
    df.rename(columns={'id_code': 'image_id'}, inplace=True)

# Build image paths
df['image_path'] = df['image_id'].apply(
    lambda x: str(TRAIN_IMAGES / f"{x}.png")
)

# Label mapping
GRADE_MAP = {0: 'No DR', 1: 'Mild', 2: 'Moderate', 3: 'Severe', 4: 'Proliferative'}
GRADE_COLORS = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c', '#8e44ad']
NUM_CLASSES = 5

df['grade_name'] = df['diagnosis'].map(GRADE_MAP)

print(f"\n✅ Dataset loaded: {len(df)} images")
print(f"\nClass distribution:")
for g in range(NUM_CLASSES):
    count = (df['diagnosis'] == g).sum()
    print(f"  Grade {g} ({GRADE_MAP[g]}): {count} ({100*count/len(df):.1f}%)")


## 🧹 Step 6 — Data Cleaning
Removes corrupt, unreadable, black, and blurry images.

In [ ]:
# — Step 6: Data Cleaning —
import cv2
import numpy as np
from tqdm import tqdm
import time

t0 = time.time()

def check_image(path):
    """Returns True if image is valid and usable."""
    try:
        img = cv2.imread(str(path))
        if img is None:
            return False, "unreadable"
        h, w = img.shape[:2]
        if h < 50 or w < 50:
            return False, "too_small"
        # Check if mostly black/white (corrupt)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        if gray.std() < 5:
            return False, "blank"
        # Check if too blurry
        lap_var = cv2.Laplacian(gray, cv2.CV_64F).var()
        if lap_var < 5:
            return False, "blurry"
        # Check for abnormal color (mostly blue/noise)
        b, g, r = cv2.split(img)
        if b.mean() > 200 and g.mean() < 50 and r.mean() < 50:
            return False, "blue_noise"
        return True, "ok"
    except Exception:
        return False, "error"

print("🔍 Scanning images for quality issues...")
results = []
for _, row in tqdm(df.iterrows(), total=len(df), desc="Checking"):
    valid, reason = check_image(row['image_path'])
    results.append({'valid': valid, 'reason': reason})

results_df = pd.DataFrame(results)
df['valid'] = results_df['valid'].values
df['reject_reason'] = results_df['reason'].values

# Report
rejected = df[~df['valid']]
if len(rejected) > 0:
    print(f"\n⚠️ Rejected {len(rejected)} images:")
    print(rejected['reject_reason'].value_counts().to_string())
else:
    print("\n✅ All images passed quality check")

# Keep only valid images
df_clean = df[df['valid']].copy().reset_index(drop=True)
df_clean.drop(columns=['valid', 'reject_reason'], inplace=True)

print(f"\n✅ Cleaning done in {time.time()-t0:.1f}s")
print(f"   Kept: {len(df_clean)} images (removed {len(df)-len(df_clean)})")
df = df_clean


## 📊 Step 7 — Exploratory Data Analysis (EDA)
Class distribution, sample images, brightness & size analysis.

In [ ]:
# — Step 7: EDA —
import matplotlib.pyplot as plt
import seaborn as sns
import cv2, numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Class distribution
counts = df['diagnosis'].value_counts().sort_index()
axes[0].bar(range(NUM_CLASSES), counts.values, color=GRADE_COLORS, edgecolor='black')
axes[0].set_xticks(range(NUM_CLASSES))
axes[0].set_xticklabels([f"G{i}\n{GRADE_MAP[i]}" for i in range(NUM_CLASSES)])
axes[0].set_title("Class Distribution", fontsize=13, fontweight='bold')
axes[0].set_ylabel("Count")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold')

# 2. Binary split (DR vs No DR)
binary = [(df['diagnosis'] == 0).sum(), (df['diagnosis'] > 0).sum()]
axes[1].pie(binary, labels=['No DR', 'DR'], colors=['#2ecc71', '#e74c3c'],
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
axes[1].set_title("Binary Split (DR vs No DR)", fontsize=13, fontweight='bold')

# 3. Imbalance ratio
ratios = counts.max() / counts
axes[2].barh(range(NUM_CLASSES), ratios.values, color=GRADE_COLORS, edgecolor='black')
axes[2].set_yticks(range(NUM_CLASSES))
axes[2].set_yticklabels([f"G{i}" for i in range(NUM_CLASSES)])
axes[2].set_xlabel("Imbalance Ratio (vs majority)")
axes[2].set_title("Class Imbalance Ratios", fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig("eda_distribution.png", dpi=120, bbox_inches='tight')
plt.show()

# Sample images per grade
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for g in range(5):
    sample = df[df['diagnosis'] == g].iloc[0]
    img = cv2.cvtColor(cv2.imread(sample['image_path']), cv2.COLOR_BGR2RGB)
    axes[g].imshow(img)
    axes[g].set_title(f"Grade {g}: {GRADE_MAP[g]}", fontsize=10, color=GRADE_COLORS[g],
                      fontweight='bold')
    axes[g].axis('off')
plt.suptitle("Sample Images per Grade", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("eda_samples.png", dpi=120, bbox_inches='tight')
plt.show()

print("✅ EDA plots saved")


## ⚖️ Step 8 — Label Analysis & Class Weights
Computes class weights for handling severe imbalance.

In [ ]:
# — Step 8: Label Analysis & Class Weights —
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

labels = df['diagnosis'].values

# Compute class weights (inverse frequency)
class_weights = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=labels)
class_weights_tensor = torch.FloatTensor(class_weights).to(DEVICE)

print("Class Weights (balanced):")
for i in range(NUM_CLASSES):
    count = (labels == i).sum()
    print(f"  Grade {i} ({GRADE_MAP[i]}): weight={class_weights[i]:.4f}  "
          f"count={count}  ratio={count/len(labels)*100:.1f}%")

# Sampler weights (per-sample weight for WeightedRandomSampler)
sample_weights = class_weights[labels]
sample_weights_tensor = torch.DoubleTensor(sample_weights)

print(f"\n✅ Class weights computed")
print(f"   Max imbalance ratio: {class_weights.max()/class_weights.min():.1f}x")


## 🖼️ Step 9 — Medical-Grade Preprocessing
Ben Graham's method + Circular Cropping + CLAHE + Green Channel Enhancement.

In [ ]:
# — Step 9: Medical-Grade Preprocessing —
import cv2
import numpy as np

IMG_SIZE = 384  # Base resolution for RTX 2050

def _make_circular_mask(img):
    """Create circular mask for fundus images."""
    h, w = img.shape[:2]
    center = (w // 2, h // 2)
    radius = min(h, w) // 2
    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.circle(mask, center, radius, 255, -1)
    return mask

def _clahe_lab(img):
    """Apply CLAHE on L channel in LAB color space."""
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    lab = cv2.merge([l, a, b])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

def _ben_graham(img, sigma=10):
    """Ben Graham's preprocessing: 4*img - 4*blur + 128."""
    blur = cv2.GaussianBlur(img.astype(np.float32), (0, 0), sigma)
    enhanced = 4.0 * img.astype(np.float32) - 4.0 * blur + 128.0
    return np.clip(enhanced, 0, 255).astype(np.uint8)

def _crop_fundus(img):
    """Crop black borders around fundus."""
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    _, thresh = cv2.threshold(gray, 15, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        cnt = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(cnt)
        # Add small margin
        margin = 5
        x, y = max(0, x - margin), max(0, y - margin)
        w, h = min(img.shape[1] - x, w + 2*margin), min(img.shape[0] - y, h + 2*margin)
        img = img[y:y+h, x:x+w]
    return img

def preprocess_fundus(image_path, size=IMG_SIZE):
    """Full preprocessing pipeline."""
    # Read
    img = cv2.imread(str(image_path))
    if img is None:
        raise ValueError(f"Cannot read: {image_path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Crop black borders
    img = _crop_fundus(img)

    # Resize preserving aspect ratio with padding
    h, w = img.shape[:2]
    scale = size / max(h, w)
    new_h, new_w = int(h * scale), int(w * scale)
    img = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)

    # Pad to square
    canvas = np.zeros((size, size, 3), dtype=np.uint8)
    y_off = (size - new_h) // 2
    x_off = (size - new_w) // 2
    canvas[y_off:y_off+new_h, x_off:x_off+new_w] = img
    img = canvas

    # Circular mask
    mask = _make_circular_mask(img)
    img = cv2.bitwise_and(img, img, mask=mask)
    bg_fill = np.where(mask[:, :, None] == 0, 128, 0).astype(np.uint8)
    img = img + bg_fill

    # Ben Graham enhancement
    img = _ben_graham(img)

    # CLAHE
    img = _clahe_lab(img)

    return img.astype(np.uint8)

# Test preprocessing
print("Testing preprocessing on sample images...")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for g in range(5):
    sample = df[df['diagnosis'] == g].iloc[0]
    orig = cv2.cvtColor(cv2.imread(sample['image_path']), cv2.COLOR_BGR2RGB)
    proc = preprocess_fundus(sample['image_path'])
    axes[0][g].imshow(orig); axes[0][g].set_title(f"Original G{g}"); axes[0][g].axis('off')
    axes[1][g].imshow(proc); axes[1][g].set_title(f"Processed G{g}"); axes[1][g].axis('off')
plt.suptitle("Ben Graham Preprocessing Pipeline", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("preprocessing_demo.png", dpi=120, bbox_inches='tight')
plt.show()

print(f"✅ preprocess_fundus() defined (output: {IMG_SIZE}×{IMG_SIZE})")


## 🔵 Step 10 — Fundus Image Verifier (Multi-Stage Validation)
Additional validation ensuring only valid retinal fundus images are processed.

In [ ]:
# — Step 10: Fundus Image Verifier —
import cv2
import numpy as np

def verify_fundus(image_path):
    """Multi-stage fundus image validation."""
    img = cv2.imread(str(image_path))
    if img is None:
        return False, "unreadable"

    h, w = img.shape[:2]

    # 1. Size check
    if h < 100 or w < 100:
        return False, "too_small"

    # 2. Color analysis — fundus images are predominantly red/orange
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    # Check if image has some warm color content
    warm_mask = ((hsv[:,:,0] < 30) | (hsv[:,:,0] > 160)) & (hsv[:,:,1] > 30)
    warm_ratio = warm_mask.sum() / (h * w)

    # 3. Circular FOV check — fundus images have circular field of view
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, binary = cv2.threshold(gray, 15, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return False, "no_content"

    largest = max(contours, key=cv2.contourArea)
    area = cv2.contourArea(largest)
    perimeter = cv2.arcLength(largest, True)

    # Circularity metric
    if perimeter > 0:
        circularity = 4 * np.pi * area / (perimeter * perimeter)
    else:
        circularity = 0

    # 4. Brightness check
    mean_brightness = gray.mean()
    if mean_brightness < 10:
        return False, "too_dark"
    if mean_brightness > 245:
        return False, "too_bright"

    # 5. Vascular structure detection (should have edges/vessels)
    edges = cv2.Canny(gray, 30, 100)
    edge_density = edges.sum() / (255 * h * w)
    if edge_density < 0.005:
        return False, "no_structure"

    return True, "valid"

# Run verification
from tqdm import tqdm

print("🔍 Running fundus image verification...")
verify_results = []
for _, row in tqdm(df.iterrows(), total=len(df), desc="Verifying"):
    valid, reason = verify_fundus(row['image_path'])
    verify_results.append(valid)

invalid_count = sum(1 for v in verify_results if not v)
print(f"\n✅ Verification complete: {len(df) - invalid_count}/{len(df)} images passed")

if invalid_count > 0:
    df = df[verify_results].reset_index(drop=True)
    print(f"   Removed {invalid_count} invalid images")


## ✂️ Step 11 — Train / Validation / Test Split
Stratified split preserving class distribution. Test set is held out completely.

In [ ]:
# — Step 11: Train / Validation / Test Split —
from sklearn.model_selection import train_test_split
import numpy as np

SEED = 42
np.random.seed(SEED)

# First split: 80% train+val, 20% test
df_train_val, df_test = train_test_split(
    df, test_size=0.10, random_state=SEED, stratify=df['diagnosis']
)

# Second split: 90% train, 10% val (from train+val)
df_train, df_val = train_test_split(
    df_train_val, test_size=0.1111, random_state=SEED, stratify=df_train_val['diagnosis']
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

print(f"✅ Split complete:")
print(f"   Train: {len(df_train)}  Val: {len(df_val)}  Test: {len(df_test)}")

# Show distribution per split
for name, _df in [("Train", df_train), ("Val", df_val), ("Test", df_test)]:
    dist = _df['diagnosis'].value_counts().sort_index()
    parts = "  ".join(f"G{g}={dist.get(g,0)}" for g in range(5))
    print(f"   {name}: {parts}")


## 🎨 Step 12 — Medical-Grade Data Augmentation & Dataset
Controlled augmentation safe for medical images + custom PyTorch Dataset.

In [ ]:
# — Step 12: Data Augmentation & Dataset —
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torch
from torch.utils.data import Dataset
import cv2
import numpy as np

# — Augmentation pipelines —
def get_train_transforms(img_size):
    return A.Compose([
        A.RandomResizedCrop(height=img_size, width=img_size, scale=(0.8, 1.0), ratio=(0.9, 1.1)),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Rotate(limit=15, p=0.5, border_mode=cv2.BORDER_CONSTANT, value=0),
        A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.3),
        A.CLAHE(clip_limit=2.0, p=0.2),
        A.GaussNoise(var_limit=(5.0, 20.0), p=0.2),
        A.CoarseDropout(max_holes=4, max_height=20, max_width=20, p=0.2),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])

def get_val_transforms(img_size):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])

# — Dataset class —
class DRDataset(Dataset):
    def __init__(self, dataframe, transform=None, preprocess=True, img_size=384):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.preprocess = preprocess
        self.img_size = img_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = row['image_path']
        label = row['diagnosis']

        # Load and preprocess
        if self.preprocess:
            try:
                image = preprocess_fundus(path, size=self.img_size)
            except Exception:
                # Fallback: just resize
                image = cv2.imread(str(path))
                image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                image = cv2.resize(image, (self.img_size, self.img_size))
        else:
            image = cv2.imread(str(path))
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            image = cv2.resize(image, (self.img_size, self.img_size))

        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']

        return image, torch.tensor(label, dtype=torch.long)

print("✅ Dataset class and augmentation pipelines defined")
print(f"   Train augmentations: RandomResizedCrop, Flip, Rotate, BrightnessContrast, CLAHE, Noise, Dropout")
print(f"   Val/Test augmentations: Resize, Normalize only")


## 🔄 Step 13 — Stratified K-Fold Setup
Sets up 5-fold cross-validation on training data only.

In [ ]:
# — Step 13: Stratified K-Fold —
from sklearn.model_selection import StratifiedKFold

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

folds = []
for fold_idx, (train_idx, val_idx) in enumerate(skf.split(df_train, df_train['diagnosis'])):
    fold_train = df_train.iloc[train_idx].reset_index(drop=True)
    fold_val = df_train.iloc[val_idx].reset_index(drop=True)
    folds.append((fold_train, fold_val))
    print(f"  Fold {fold_idx+1}: Train={len(fold_train)}, Val={len(fold_val)}")

print(f"\n✅ {N_FOLDS}-Fold StratifiedKFold created")
print(f"   Class distribution preserved in each fold")


## 🔧 Step 14 — DataLoader Creation
Optimized batch sizes for RTX 2050 (4GB VRAM) with WeightedRandomSampler.

In [ ]:
# — Step 14: DataLoader Creation —
from torch.utils.data import DataLoader, WeightedRandomSampler

# RTX 2050 (4GB) optimized batch sizes
BATCH_SIZES = {
    224: 16,   # Phase 1
    384: 8,    # Phase 2
    512: 4,    # Phase 3
}
NUM_WORKERS = 4  # i5-12450H has 8 cores

def create_dataloaders(fold_train_df, fold_val_df, img_size, batch_size=None):
    """Create train and val dataloaders with WeightedRandomSampler."""
    if batch_size is None:
        batch_size = BATCH_SIZES.get(img_size, 8)

    # Train dataset
    train_ds = DRDataset(fold_train_df, transform=get_train_transforms(img_size),
                         preprocess=True, img_size=img_size)
    # Val dataset
    val_ds = DRDataset(fold_val_df, transform=get_val_transforms(img_size),
                       preprocess=True, img_size=img_size)

    # Weighted sampler for class imbalance
    train_labels = fold_train_df['diagnosis'].values
    cw = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=train_labels)
    sample_w = torch.DoubleTensor([cw[l] for l in train_labels])
    sampler = WeightedRandomSampler(sample_w, len(sample_w), replacement=True)

    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler,
                              num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size * 2, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)

    return train_loader, val_loader

# Test dataloader (created once, used at the end)
def create_test_loader(test_df, img_size, batch_size=None):
    if batch_size is None:
        batch_size = BATCH_SIZES.get(img_size, 8)
    test_ds = DRDataset(test_df, transform=get_val_transforms(img_size),
                        preprocess=True, img_size=img_size)
    return DataLoader(test_ds, batch_size=batch_size * 2, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True)

print(f"✅ DataLoader factory defined")
print(f"   Batch sizes: {BATCH_SIZES}")
print(f"   Workers: {NUM_WORKERS}")
print(f"   WeightedRandomSampler: ENABLED")


## 🧠 Step 15 — Model Architecture
EfficientNetV2-B1 backbone with custom classification head.

In [ ]:
# — Step 15: Model Architecture —
import torch
import torch.nn as nn
import timm

BACKBONE = 'tf_efficientnetv2_b1'

class DRModel(nn.Module):
    def __init__(self, backbone=BACKBONE, num_classes=NUM_CLASSES, pretrained=True, drop_rate=0.5):
        super().__init__()
        # Load pretrained backbone
        self.backbone = timm.create_model(backbone, pretrained=pretrained, num_classes=0)
        in_features = self.backbone.num_features

        # Custom classification head
        self.head = nn.Sequential(
            nn.BatchNorm1d(in_features),
            nn.Linear(in_features, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(drop_rate),
            nn.Linear(256, num_classes)
        )

        # Initialize head weights
        for m in self.head.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        features = self.backbone(x)  # Global average pooling built-in
        return self.head(features)

    def freeze_backbone(self):
        """Freeze all backbone parameters."""
        for param in self.backbone.parameters():
            param.requires_grad = False
        print("  🧊 Backbone frozen")

    def unfreeze_backbone(self, num_blocks=None):
        """Unfreeze backbone (all or last N blocks)."""
        if num_blocks is None:
            # Unfreeze all
            for param in self.backbone.parameters():
                param.requires_grad = True
            print("  🔥 Full backbone unfrozen")
        else:
            # Unfreeze last N blocks
            blocks = list(self.backbone.blocks) if hasattr(self.backbone, 'blocks') else []
            if not blocks:
                # Fallback: unfreeze all
                for param in self.backbone.parameters():
                    param.requires_grad = True
                print(f"  🔥 Full backbone unfrozen (no block structure found)")
                return

            # Freeze all first
            for param in self.backbone.parameters():
                param.requires_grad = False
            # Unfreeze last N blocks
            for block in blocks[-num_blocks:]:
                for param in block.parameters():
                    param.requires_grad = True
            # Always unfreeze final norm
            if hasattr(self.backbone, 'bn2'):
                for param in self.backbone.bn2.parameters():
                    param.requires_grad = True
            if hasattr(self.backbone, 'conv_head'):
                for param in self.backbone.conv_head.parameters():
                    param.requires_grad = True
            print(f"  🔥 Last {num_blocks} blocks unfrozen")

# Test model creation
model = DRModel()
model.to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n✅ Model: {BACKBONE}")
print(f"   Total params: {total_params:,}")
print(f"   Trainable params: {trainable_params:,}")
print(f"   Head features: {model.backbone.num_features} → 256 → {NUM_CLASSES}")

# Test forward pass
with torch.no_grad():
    dummy = torch.randn(2, 3, 384, 384).to(DEVICE)
    out = model(dummy)
    print(f"   Test forward: input={dummy.shape} → output={out.shape}")

# Clear test tensors
del dummy, out
torch.cuda.empty_cache() if torch.cuda.is_available() else None


## ⚙️ Step 16 — Loss Function + Optimizer + Scheduler
Hybrid loss (CrossEntropy + Focal), AdamW optimizer, Cosine Annealing.

In [ ]:
# — Step 16: Loss + Optimizer + Scheduler —
import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    """Focal Loss for handling class imbalance."""
    def __init__(self, gamma=2.0, alpha=None, label_smoothing=0.0):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        if self.label_smoothing > 0:
            n_classes = inputs.size(1)
            smooth_targets = torch.zeros_like(inputs).scatter_(
                1, targets.unsqueeze(1), 1.0
            )
            smooth_targets = smooth_targets * (1 - self.label_smoothing) + \
                           self.label_smoothing / n_classes
            log_probs = F.log_softmax(inputs, dim=1)
            ce_loss = -(smooth_targets * log_probs).sum(dim=1)
        else:
            ce_loss = F.cross_entropy(inputs, targets, reduction='none')

        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss

        if self.alpha is not None:
            alpha_t = self.alpha[targets]
            focal_loss = alpha_t * focal_loss

        return focal_loss.mean()

class HybridLoss(nn.Module):
    """0.5 * Weighted CE + 0.5 * Focal Loss."""
    def __init__(self, class_weights, gamma=2.0, label_smoothing=0.05):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=label_smoothing)
        self.focal = FocalLoss(gamma=gamma, alpha=class_weights, label_smoothing=label_smoothing)

    def forward(self, inputs, targets):
        return 0.5 * self.ce(inputs, targets) + 0.5 * self.focal(inputs, targets)

def create_optimizer_scheduler(model, lr, epochs, steps_per_epoch):
    """Create AdamW optimizer and Cosine Annealing scheduler."""
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,
        weight_decay=1e-4
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=epochs * steps_per_epoch,
        eta_min=1e-6
    )
    return optimizer, scheduler

print("✅ Loss functions defined:")
print("   HybridLoss = 0.5 * WeightedCE + 0.5 * FocalLoss(γ=2)")
print("   Label smoothing: 0.05")
print("   Optimizer: AdamW (weight_decay=1e-4)")
print("   Scheduler: CosineAnnealingLR")


## 🏋️ Step 17 — Training Engine
Core training and validation functions with mixed precision support.

In [ ]:
# — Step 17: Training Engine —
import torch
import numpy as np
from tqdm import tqdm
from sklearn.metrics import cohen_kappa_score
from scipy.stats import spearmanr

def qwk_score(y_true, y_pred):
    """Quadratic Weighted Kappa."""
    return cohen_kappa_score(y_true, y_pred, weights='quadratic')

scaler = torch.amp.GradScaler('cuda') if DEVICE.type == 'cuda' else None

def train_one_epoch(model, loader, criterion, optimizer, scheduler, grad_accum=1):
    """Train for one epoch with mixed precision and gradient accumulation."""
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []

    optimizer.zero_grad()
    pbar = tqdm(loader, desc="  Training", leave=False)

    for batch_idx, (images, labels) in enumerate(pbar):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        # Mixed precision forward
        if DEVICE.type == 'cuda':
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels) / grad_accum
            scaler.scale(loss).backward()

            if (batch_idx + 1) % grad_accum == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                if scheduler:
                    scheduler.step()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels) / grad_accum
            loss.backward()

            if (batch_idx + 1) % grad_accum == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()
                if scheduler:
                    scheduler.step()

        running_loss += loss.item() * grad_accum
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

        pbar.set_postfix({'loss': f'{running_loss/(batch_idx+1):.4f}'})

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    avg_loss = running_loss / len(loader)
    accuracy = (all_preds == all_labels).mean()
    qwk = qwk_score(all_labels, all_preds)

    return avg_loss, accuracy, qwk

@torch.no_grad()
def validate(model, loader, criterion):
    """Validate model."""
    model.eval()
    running_loss = 0.0
    all_preds, all_labels, all_probs = [], [], []

    for images, labels in tqdm(loader, desc="  Validating", leave=False):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        if DEVICE.type == 'cuda':
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels)
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)

        running_loss += loss.item()
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs)

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    avg_loss = running_loss / len(loader)
    accuracy = (all_preds == all_labels).mean()
    qwk = qwk_score(all_labels, all_preds)

    return avg_loss, accuracy, qwk, all_probs, all_preds

def save_checkpoint(model, optimizer, scheduler, epoch, val_loss, val_qwk, history, path):
    """Save full training state for resume."""
    torch.save({
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict() if scheduler else None,
        'val_loss': val_loss,
        'val_qwk': val_qwk,
        'history': history,
        'backbone': BACKBONE,
        'img_size': IMG_SIZE,
        'scaler_state': scaler.state_dict() if scaler else None,
    }, path)

print("✅ Training engine ready")
print("   Mixed precision (FP16): " + ("ENABLED" if DEVICE.type == 'cuda' else "DISABLED"))
print("   Gradient clipping: max_norm=1.0")


## 💾 Step 18 — Checkpoint & Resume System
Full recovery support with epoch, batch, and phase tracking.

In [ ]:
# — Step 18: Checkpoint & Resume System —
from pathlib import Path

ARTIFACT_DIR = Path("./artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)
BEST_CKPT = ARTIFACT_DIR / "best_model.pt"

class TrainingTracker:
    """Track training progress across phases and folds."""
    def __init__(self):
        self.history = {
            'phase': [],
            'fold': [],
            'epoch': [],
            'train_loss': [],
            'val_loss': [],
            'train_qwk': [],
            'val_qwk': [],
            'train_acc': [],
            'val_acc': [],
            'lr': [],
        }
        self.best_qwk = -1.0
        self.best_epoch = -1
        self.best_phase = ""
        self.oof_predictions = {}
        self.oof_labels = {}

    def log(self, phase, fold, epoch, train_loss, val_loss, train_qwk, val_qwk,
            train_acc, val_acc, lr):
        self.history['phase'].append(phase)
        self.history['fold'].append(fold)
        self.history['epoch'].append(epoch)
        self.history['train_loss'].append(train_loss)
        self.history['val_loss'].append(val_loss)
        self.history['train_qwk'].append(train_qwk)
        self.history['val_qwk'].append(val_qwk)
        self.history['train_acc'].append(train_acc)
        self.history['val_acc'].append(val_acc)
        self.history['lr'].append(lr)

    def update_best(self, qwk, epoch, phase, model, path):
        if qwk > self.best_qwk:
            self.best_qwk = qwk
            self.best_epoch = epoch
            self.best_phase = phase
            torch.save({'model_state': model.state_dict(), 'qwk': qwk}, path)
            return True
        return False

    def store_oof(self, fold, preds, labels):
        self.oof_predictions[fold] = preds
        self.oof_labels[fold] = labels

tracker = TrainingTracker()
print("✅ Checkpoint & resume system ready")
print(f"   Artifacts directory: {ARTIFACT_DIR}")
print(f"   Best model path: {BEST_CKPT}")


## ⏹️ Step 19 — Early Stopping

In [ ]:
# — Step 19: Early Stopping —

class EarlyStopping:
    """Early stopping based on validation QWK."""
    def __init__(self, patience=5, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        self.should_stop = False

    def __call__(self, score):
        if self.best_score is None:
            self.best_score = score
        elif score < self.best_score + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
                print(f"  ⏹️ Early stopping triggered (patience={self.patience})")
        else:
            self.best_score = score
            self.counter = 0

    def reset(self):
        self.counter = 0
        self.best_score = None
        self.should_stop = False

print("✅ Early stopping defined (patience=5, min_delta=0.001)")


## 🚀 Step 20 — Full Training Pipeline (3-Phase Strategy)

| Phase | Resolution | Epochs | Backbone | LR |
|-------|-----------|--------|----------|-----|
| 1 - Feature Learning | 224×224 | 15 | Frozen | 1e-3 |
| 2 - Representation | 384×384 | 25 | Partial unfreeze | 3e-4 |
| 3 - Fine-Tuning | 384×384 | 15 | Full unfreeze | 1e-4 |

> ⚠️ **This cell trains the full model. Expected time: 3-5 hours on RTX 2050.**
> Resolution 512 is reduced to 384 to fit in 4GB VRAM.


In [ ]:
# — Step 20: Full Training Pipeline —
import gc, time

# Phase configuration (optimized for 4GB VRAM)
PHASES = [
    {
        'name': 'Phase 1: Head Only (Backbone Frozen)',
        'img_size': 224,
        'epochs': 15,
        'lr': 1e-3,
        'batch_size': 16,
        'grad_accum': 1,
        'freeze': 'all',       # freeze entire backbone
        'patience': 5,
    },
    {
        'name': 'Phase 2: Progressive Unfreeze',
        'img_size': 384,
        'epochs': 25,
        'lr': 3e-4,
        'batch_size': 6,
        'grad_accum': 2,       # effective batch = 12
        'freeze': 'partial',   # unfreeze last 4 blocks
        'patience': 5,
    },
    {
        'name': 'Phase 3: Full Fine-Tuning',
        'img_size': 384,       # 512 won't fit in 4GB, use 384
        'epochs': 15,
        'lr': 1e-4,
        'batch_size': 4,
        'grad_accum': 4,       # effective batch = 16
        'freeze': 'none',      # unfreeze all
        'patience': 5,
    },
]

# We train on a SINGLE fold for efficiency (fold 0), then validate
# Full 5-fold CV is in Step 21

print("=" * 70)
print("  TRAINING PIPELINE — RTX 2050 (4GB) Optimized")
print("=" * 70)

# Reset model
model = DRModel(backbone=BACKBONE, pretrained=True, drop_rate=0.5)
model.to(DEVICE)

global_best_qwk = -1.0
all_histories = []

for phase_idx, phase in enumerate(PHASES):
    print(f"\n{'='*70}")
    print(f"  {phase['name']}")
    print(f"  Resolution: {phase['img_size']}  Epochs: {phase['epochs']}  "
          f"LR: {phase['lr']}  Batch: {phase['batch_size']}×{phase['grad_accum']}")
    print(f"{'='*70}")

    # Configure backbone freezing
    if phase['freeze'] == 'all':
        model.freeze_backbone()
    elif phase['freeze'] == 'partial':
        model.unfreeze_backbone(num_blocks=4)
    else:
        model.unfreeze_backbone()

    # Print trainable params
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"  Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

    # Create dataloaders for fold 0
    fold_train_df, fold_val_df = folds[0]
    train_loader, val_loader = create_dataloaders(
        fold_train_df, fold_val_df,
        img_size=phase['img_size'],
        batch_size=phase['batch_size']
    )

    # Create loss, optimizer, scheduler
    criterion = HybridLoss(class_weights_tensor).to(DEVICE)
    optimizer, scheduler = create_optimizer_scheduler(
        model, lr=phase['lr'],
        epochs=phase['epochs'],
        steps_per_epoch=len(train_loader) // phase['grad_accum']
    )

    early_stop = EarlyStopping(patience=phase['patience'])
    phase_best_qwk = -1.0
    t_phase = time.time()

    for epoch in range(phase['epochs']):
        t_epoch = time.time()
        current_lr = optimizer.param_groups[0]['lr']

        # Train
        train_loss, train_acc, train_qwk = train_one_epoch(
            model, train_loader, criterion, optimizer, scheduler,
            grad_accum=phase['grad_accum']
        )

        # Validate
        val_loss, val_acc, val_qwk, val_probs, val_preds = validate(
            model, val_loader, criterion
        )

        # Track
        tracker.log(
            phase=phase['name'], fold=0, epoch=epoch+1,
            train_loss=train_loss, val_loss=val_loss,
            train_qwk=train_qwk, val_qwk=val_qwk,
            train_acc=train_acc, val_acc=val_acc,
            lr=current_lr
        )

        # Best model
        is_best = tracker.update_best(val_qwk, epoch+1, phase['name'], model, BEST_CKPT)
        best_marker = " ⭐ NEW BEST" if is_best else ""

        elapsed = time.time() - t_epoch
        print(f"  Epoch {epoch+1:2d}/{phase['epochs']} | "
              f"Loss: {train_loss:.4f}/{val_loss:.4f} | "
              f"Acc: {train_acc:.3f}/{val_acc:.3f} | "
              f"QWK: {train_qwk:.4f}/{val_qwk:.4f} | "
              f"LR: {current_lr:.2e} | {elapsed:.0f}s{best_marker}")

        if is_best:
            phase_best_qwk = val_qwk

        # Early stopping
        early_stop(val_qwk)
        if early_stop.should_stop:
            break

        # Memory cleanup
        torch.cuda.empty_cache() if torch.cuda.is_available() else None

    phase_time = time.time() - t_phase
    print(f"\n  ✅ {phase['name']} complete in {phase_time/60:.1f} min")
    print(f"     Best Val QWK this phase: {phase_best_qwk:.4f}")
    print(f"     Global Best Val QWK: {tracker.best_qwk:.4f}")

    # Clear memory between phases
    del train_loader, val_loader, optimizer, scheduler, criterion
    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

print(f"\n{'='*70}")
print(f"  ✅ TRAINING COMPLETE")
print(f"  Best QWK: {tracker.best_qwk:.4f} (Epoch {tracker.best_epoch}, {tracker.best_phase})")
print(f"{'='*70}")


## 🔁 Step 21 — Cross-Validation Training (5-Fold)
> **Optional but recommended.** This runs the best phase config across all 5 folds.
> Each fold trains only Phase 1 + Phase 2 for time efficiency.
> Skip this cell if you want to save time and use the single-fold model from Step 20.


In [ ]:
# — Step 21: 5-Fold Cross-Validation (Optional) —
# Set to True to run full 5-fold CV (takes ~15-20 hours on RTX 2050)
RUN_FULL_CV = False

if RUN_FULL_CV:
    fold_qwks = []
    oof_all_preds = np.zeros(len(df_train))
    oof_all_labels = np.zeros(len(df_train))

    for fold_idx in range(N_FOLDS):
        print(f"\n{'='*60}")
        print(f"  FOLD {fold_idx + 1}/{N_FOLDS}")
        print(f"{'='*60}")

        fold_train_df, fold_val_df = folds[fold_idx]

        # Fresh model per fold
        fold_model = DRModel(backbone=BACKBONE, pretrained=True).to(DEVICE)

        # Phase 1: Head only
        fold_model.freeze_backbone()
        train_loader, val_loader = create_dataloaders(fold_train_df, fold_val_df, 224, 16)
        criterion = HybridLoss(class_weights_tensor).to(DEVICE)
        optimizer, scheduler = create_optimizer_scheduler(fold_model, 1e-3, 10, len(train_loader))
        es = EarlyStopping(patience=3)

        for epoch in range(10):
            train_loss, train_acc, train_qwk = train_one_epoch(
                fold_model, train_loader, criterion, optimizer, scheduler)
            val_loss, val_acc, val_qwk, val_probs, val_preds = validate(
                fold_model, val_loader, criterion)
            print(f"  F{fold_idx+1} P1 E{epoch+1}: QWK={val_qwk:.4f} Acc={val_acc:.3f}")
            es(val_qwk)
            if es.should_stop:
                break

        # Phase 2: Partial unfreeze
        fold_model.unfreeze_backbone(num_blocks=4)
        train_loader, val_loader = create_dataloaders(fold_train_df, fold_val_df, 384, 6)
        optimizer, scheduler = create_optimizer_scheduler(fold_model, 3e-4, 15, len(train_loader)//2)
        es.reset()
        best_fold_qwk = -1

        for epoch in range(15):
            train_loss, train_acc, train_qwk = train_one_epoch(
                fold_model, train_loader, criterion, optimizer, scheduler, grad_accum=2)
            val_loss, val_acc, val_qwk, val_probs, val_preds = validate(
                fold_model, val_loader, criterion)
            if val_qwk > best_fold_qwk:
                best_fold_qwk = val_qwk
            print(f"  F{fold_idx+1} P2 E{epoch+1}: QWK={val_qwk:.4f} Acc={val_acc:.3f}")
            es(val_qwk)
            if es.should_stop:
                break

        fold_qwks.append(best_fold_qwk)
        tracker.store_oof(fold_idx, val_preds, fold_val_df['diagnosis'].values)
        print(f"  ✅ Fold {fold_idx+1} Best QWK: {best_fold_qwk:.4f}")

        del fold_model, train_loader, val_loader, optimizer, scheduler
        gc.collect()
        torch.cuda.empty_cache() if torch.cuda.is_available() else None

    print(f"\n{'='*60}")
    print(f"  5-FOLD CV RESULTS")
    print(f"  Per-fold QWK: {[f'{q:.4f}' for q in fold_qwks]}")
    print(f"  Mean QWK: {np.mean(fold_qwks):.4f} ± {np.std(fold_qwks):.4f}")
    print(f"{'='*60}")
else:
    print("⏩ Skipping 5-fold CV (RUN_FULL_CV = False)")
    print("   Using single-fold model from Step 20")
    # Store OOF from fold 0
    _, fold_val_df_0 = folds[0]
    val_loader_temp = create_test_loader(fold_val_df_0, 384)
    criterion_temp = HybridLoss(class_weights_tensor).to(DEVICE)

    # Load best model
    if BEST_CKPT.exists():
        ckpt = torch.load(BEST_CKPT, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt['model_state'])
        print(f"   Loaded best model (QWK={ckpt['qwk']:.4f})")

    _, _, _, oof_probs, oof_preds = validate(model, val_loader_temp, criterion_temp)
    tracker.store_oof(0, oof_preds, fold_val_df_0['diagnosis'].values)
    del val_loader_temp, criterion_temp


## 🔄 Step 22 — Test-Time Augmentation (TTA)

In [ ]:
# — Step 22: Test-Time Augmentation —
import albumentations as A
from albumentations.pytorch import ToTensorV2

def get_tta_transforms(img_size):
    """Return list of TTA transforms."""
    base_norm = A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    return [
        # Original
        A.Compose([A.Resize(img_size, img_size), base_norm, ToTensorV2()]),
        # Horizontal flip
        A.Compose([A.Resize(img_size, img_size), A.HorizontalFlip(p=1.0), base_norm, ToTensorV2()]),
        # Vertical flip
        A.Compose([A.Resize(img_size, img_size), A.VerticalFlip(p=1.0), base_norm, ToTensorV2()]),
        # Brightness+
        A.Compose([A.Resize(img_size, img_size),
                   A.RandomBrightnessContrast(brightness_limit=(0.05, 0.05), contrast_limit=0, p=1.0),
                   base_norm, ToTensorV2()]),
    ]

@torch.no_grad()
def predict_with_tta(model, test_df, img_size=384):
    """Run inference with TTA and average predictions."""
    model.eval()
    tta_transforms = get_tta_transforms(img_size)
    all_probs = []

    for tta_idx, transform in enumerate(tta_transforms):
        ds = DRDataset(test_df, transform=transform, preprocess=True, img_size=img_size)
        loader = DataLoader(ds, batch_size=8, shuffle=False, num_workers=NUM_WORKERS,
                          pin_memory=True)

        batch_probs = []
        for images, _ in tqdm(loader, desc=f"  TTA {tta_idx+1}/{len(tta_transforms)}", leave=False):
            images = images.to(DEVICE, non_blocking=True)
            if DEVICE.type == 'cuda':
                with torch.amp.autocast('cuda'):
                    outputs = model(images)
            else:
                outputs = model(images)
            probs = torch.softmax(outputs, dim=1).cpu().numpy()
            batch_probs.append(probs)

        all_probs.append(np.vstack(batch_probs))

    # Average TTA predictions
    avg_probs = np.mean(all_probs, axis=0)
    return avg_probs

print("✅ TTA defined (4 augmentations: original, h-flip, v-flip, brightness)")


## 🎯 Step 23 — Threshold Optimization
Optimizes decision thresholds using OOF predictions to maximize QWK.

In [ ]:
# — Step 23: Threshold Optimization —
from scipy.optimize import minimize
import numpy as np

def optimize_thresholds(probs, true_labels):
    """Find optimal thresholds that maximize QWK."""
    def _neg_qwk(thresholds):
        thresholds = sorted(thresholds)
        preds = np.digitize(probs @ np.arange(NUM_CLASSES), thresholds)
        preds = np.clip(preds, 0, NUM_CLASSES - 1)
        return -qwk_score(true_labels, preds)

    # Initial thresholds (evenly spaced)
    init_thresholds = [0.5, 1.5, 2.5, 3.5]

    result = minimize(
        _neg_qwk,
        x0=init_thresholds,
        method='Nelder-Mead',
        options={'maxiter': 10000, 'xatol': 1e-6}
    )

    optimal_thresholds = sorted(result.x)
    return optimal_thresholds

def apply_thresholds(probs, thresholds):
    """Apply optimized thresholds to convert probabilities to predictions."""
    expected = probs @ np.arange(NUM_CLASSES)
    preds = np.digitize(expected, thresholds)
    return np.clip(preds, 0, NUM_CLASSES - 1)

# Optimize using OOF predictions
if 0 in tracker.oof_predictions:
    oof_preds_raw = tracker.oof_predictions[0]
    oof_labels_raw = tracker.oof_labels[0]

    # Get probabilities for threshold optimization
    # Re-run validation to get probabilities
    fold_val_df_0 = folds[0][1]
    val_loader_t = create_test_loader(fold_val_df_0, 384)
    criterion_t = HybridLoss(class_weights_tensor).to(DEVICE)

    if BEST_CKPT.exists():
        ckpt = torch.load(BEST_CKPT, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt['model_state'])

    _, _, _, oof_probs, _ = validate(model, val_loader_t, criterion_t)

    # Optimize
    optimal_thresholds = optimize_thresholds(oof_probs, oof_labels_raw)
    print(f"✅ Optimal thresholds: {[f'{t:.4f}' for t in optimal_thresholds]}")

    # Compare argmax vs optimized
    argmax_preds = oof_probs.argmax(axis=1)
    opt_preds = apply_thresholds(oof_probs, optimal_thresholds)

    qwk_argmax = qwk_score(oof_labels_raw, argmax_preds)
    qwk_opt = qwk_score(oof_labels_raw, opt_preds)
    print(f"   Argmax QWK: {qwk_argmax:.4f}")
    print(f"   Optimized QWK: {qwk_opt:.4f}")
    print(f"   Improvement: {qwk_opt - qwk_argmax:+.4f}")

    del val_loader_t, criterion_t
else:
    optimal_thresholds = [0.5, 1.5, 2.5, 3.5]
    print("⚠️ No OOF predictions available, using default thresholds")


## 🧪 Step 24 — Final Testing (Hold-Out Set)
Tests on the held-out test set with TTA + optimized thresholds.

In [ ]:
# — Step 24: Final Testing —
import time

print("🧪 Running final evaluation on hold-out test set...")
t0 = time.time()

# Load best model
if BEST_CKPT.exists():
    ckpt = torch.load(BEST_CKPT, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state'])
    print(f"  Loaded best model (Val QWK={ckpt['qwk']:.4f})")

# Run with TTA
test_probs = predict_with_tta(model, df_test, img_size=384)

# Apply optimized thresholds
test_preds_opt = apply_thresholds(test_probs, optimal_thresholds)
test_preds_argmax = test_probs.argmax(axis=1)
test_labels = df_test['diagnosis'].values

# Metrics
test_qwk_opt = qwk_score(test_labels, test_preds_opt)
test_qwk_argmax = qwk_score(test_labels, test_preds_argmax)
test_acc_opt = (test_preds_opt == test_labels).mean()
test_acc_argmax = (test_preds_argmax == test_labels).mean()

elapsed = time.time() - t0
print(f"\n{'='*60}")
print(f"  TEST SET RESULTS (n={len(df_test)})")
print(f"{'='*60}")
print(f"  With Optimized Thresholds:")
print(f"    QWK:      {test_qwk_opt:.4f}")
print(f"    Accuracy: {test_acc_opt:.4f}")
print(f"  With Argmax:")
print(f"    QWK:      {test_qwk_argmax:.4f}")
print(f"    Accuracy: {test_acc_argmax:.4f}")
print(f"  Time: {elapsed:.1f}s (with TTA)")
print(f"{'='*60}")


## 📈 Step 25 — Metrics & Evaluation
Confusion matrix, per-class metrics, training curves.

In [ ]:
# — Step 25: Metrics & Evaluation —
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

# Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Optimized thresholds
cm_opt = confusion_matrix(test_labels, test_preds_opt)
sns.heatmap(cm_opt, annot=True, fmt='d', cmap='Blues', ax=axes[0],
           xticklabels=[GRADE_MAP[i] for i in range(5)],
           yticklabels=[GRADE_MAP[i] for i in range(5)])
axes[0].set_title(f'Optimized Thresholds (QWK={test_qwk_opt:.4f})', fontweight='bold')
axes[0].set_ylabel('True'); axes[0].set_xlabel('Predicted')

# Argmax
cm_argmax = confusion_matrix(test_labels, test_preds_argmax)
sns.heatmap(cm_argmax, annot=True, fmt='d', cmap='Oranges', ax=axes[1],
           xticklabels=[GRADE_MAP[i] for i in range(5)],
           yticklabels=[GRADE_MAP[i] for i in range(5)])
axes[1].set_title(f'Argmax (QWK={test_qwk_argmax:.4f})', fontweight='bold')
axes[1].set_ylabel('True'); axes[1].set_xlabel('Predicted')

plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=120, bbox_inches='tight')
plt.show()

# Classification report
print("\nClassification Report (Optimized Thresholds):")
print(classification_report(test_labels, test_preds_opt,
      target_names=[GRADE_MAP[i] for i in range(5)], digits=4))

# Per-class recall
print("Per-class Recall:")
for g in range(5):
    mask = test_labels == g
    if mask.sum() > 0:
        recall = (test_preds_opt[mask] == g).mean()
        print(f"  Grade {g} ({GRADE_MAP[g]}): {recall:.4f} ({mask.sum()} samples)")

# Training curves
import pandas as pd
hist = pd.DataFrame(tracker.history)
if len(hist) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].plot(hist['train_loss'], label='Train', color='blue')
    axes[0].plot(hist['val_loss'], label='Val', color='red')
    axes[0].set_title('Loss', fontweight='bold'); axes[0].legend()
    axes[0].set_xlabel('Step')

    axes[1].plot(hist['train_qwk'], label='Train', color='blue')
    axes[1].plot(hist['val_qwk'], label='Val', color='red')
    axes[1].set_title('QWK', fontweight='bold'); axes[1].legend()
    axes[1].set_xlabel('Step')

    axes[2].plot(hist['lr'], color='green')
    axes[2].set_title('Learning Rate', fontweight='bold')
    axes[2].set_xlabel('Step')

    plt.tight_layout()
    plt.savefig("training_curves.png", dpi=120, bbox_inches='tight')
    plt.show()

print("\n✅ Evaluation complete")


## 💾 Step 26 — Model Export
Saves final model, thresholds, and configuration for deployment.

In [ ]:
# — Step 26: Model Export —
import json
from pathlib import Path

EXPORT_DIR = Path("./model_export")
EXPORT_DIR.mkdir(exist_ok=True)

# Load best model
if BEST_CKPT.exists():
    ckpt = torch.load(BEST_CKPT, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state'])

# Save model weights
model_path = EXPORT_DIR / "dr_model_final.pt"
torch.save({
    'model_state': model.state_dict(),
    'backbone': BACKBONE,
    'num_classes': NUM_CLASSES,
    'img_size': 384,
}, model_path)
print(f"✅ Model saved: {model_path} ({model_path.stat().st_size/1e6:.1f} MB)")

# Save thresholds
thresh_path = EXPORT_DIR / "thresholds.json"
with open(thresh_path, 'w') as f:
    json.dump({
        'thresholds': [float(t) for t in optimal_thresholds],
        'method': 'nelder-mead',
    }, f, indent=2)
print(f"✅ Thresholds saved: {thresh_path}")

# Save config
config = {
    'backbone': BACKBONE,
    'num_classes': NUM_CLASSES,
    'img_size': 384,
    'grade_map': GRADE_MAP,
    'preprocessing': 'ben_graham + clahe + circular_crop',
    'normalization': {'mean': [0.485, 0.456, 0.406], 'std': [0.229, 0.224, 0.225]},
    'test_qwk': float(test_qwk_opt),
    'test_accuracy': float(test_acc_opt),
    'optimal_thresholds': [float(t) for t in optimal_thresholds],
}
config_path = EXPORT_DIR / "config.json"
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)
print(f"✅ Config saved: {config_path}")

# Save label mapping
label_path = EXPORT_DIR / "label_mapping.json"
with open(label_path, 'w') as f:
    json.dump({str(k): v for k, v in GRADE_MAP.items()}, f, indent=2)
print(f"✅ Label mapping saved: {label_path}")

print(f"\n📁 Export directory: {EXPORT_DIR}")
for f in sorted(EXPORT_DIR.iterdir()):
    print(f"   {f.name} ({f.stat().st_size/1e3:.1f} KB)")


## 🔍 Step 27 — Grad-CAM++ Explainability
Visualizes where the model focuses attention for each prediction.

In [ ]:
# — Step 27: Grad-CAM++ Explainability —
import torch
import torch.nn.functional as F
import cv2
import numpy as np
import matplotlib.pyplot as plt

class GradCAMPlusPlus:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None

        # Register hooks
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, class_idx=None):
        self.model.eval()
        output = self.model(input_tensor)

        if class_idx is None:
            class_idx = output.argmax(dim=1).item()

        self.model.zero_grad()
        target = output[0, class_idx]
        target.backward()

        gradients = self.gradients[0]  # (C, H, W)
        activations = self.activations[0]  # (C, H, W)

        # Grad-CAM++ weights
        grad_2 = gradients ** 2
        grad_3 = gradients ** 3
        sum_activations = activations.sum(dim=(1, 2), keepdim=True)
        alpha = grad_2 / (2 * grad_2 + sum_activations * grad_3 + 1e-7)
        weights = (alpha * F.relu(gradients)).sum(dim=(1, 2))

        # Weighted combination
        cam = (weights[:, None, None] * activations).sum(dim=0)
        cam = F.relu(cam)
        cam = cam.cpu().numpy()

        # Normalize
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-7)
        return cam, class_idx

# Find target layer (last conv layer in backbone)
target_layer = None
for name, module in model.backbone.named_modules():
    if isinstance(module, (torch.nn.Conv2d,)):
        target_layer = module
        target_name = name

print(f"Target layer for Grad-CAM: {target_name}")
gradcam = GradCAMPlusPlus(model, target_layer)

# Visualize on test samples
fig, axes = plt.subplots(3, 5, figsize=(20, 12))

for g in range(5):
    samples = df_test[df_test['diagnosis'] == g]
    if len(samples) == 0:
        continue
    sample = samples.iloc[0]

    # Original image
    orig = preprocess_fundus(sample['image_path'], size=384)

    # Prepare input tensor
    transform = get_val_transforms(384)
    input_tensor = transform(image=orig)['image'].unsqueeze(0).to(DEVICE)

    # Generate Grad-CAM
    cam, pred_class = gradcam.generate(input_tensor)
    cam_resized = cv2.resize(cam, (384, 384))

    # Display
    axes[0][g].imshow(orig)
    axes[0][g].set_title(f"Original G{g}", fontweight='bold')
    axes[0][g].axis('off')

    axes[1][g].imshow(cam_resized, cmap='jet')
    axes[1][g].set_title(f"Attention Map", fontweight='bold')
    axes[1][g].axis('off')

    # Overlay
    heatmap = cv2.applyColorMap((cam_resized * 255).astype(np.uint8), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay = (0.6 * orig + 0.4 * heatmap).astype(np.uint8)
    axes[2][g].imshow(overlay)
    axes[2][g].set_title(f"Pred: G{pred_class} ({GRADE_MAP[pred_class]})",
                         fontweight='bold', color=GRADE_COLORS[pred_class])
    axes[2][g].axis('off')

axes[0][0].set_ylabel("Input", fontsize=12)
axes[1][0].set_ylabel("Grad-CAM++", fontsize=12)
axes[2][0].set_ylabel("Overlay", fontsize=12)
plt.suptitle("Grad-CAM++ Explainability — Model Attention per DR Grade", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("gradcam_results.png", dpi=120, bbox_inches='tight')
plt.show()

print("✅ Grad-CAM++ visualization complete")


## 🌐 Step 28 — Streamlit Deployment App
Creates a Streamlit app for single-image DR grading with heatmap visualization.

In [ ]:
# — Step 28: Streamlit App Generation —
from pathlib import Path

app_code = '''
import streamlit as st
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import cv2
import numpy as np
import json
from pathlib import Path
from PIL import Image

# ——— Configuration ———
MODEL_PATH = "model_export/dr_model_final.pt"
CONFIG_PATH = "model_export/config.json"
THRESH_PATH = "model_export/thresholds.json"

GRADE_MAP = {0: "No DR", 1: "Mild", 2: "Moderate", 3: "Severe", 4: "Proliferative"}
GRADE_COLORS = ["green", "gold", "orange", "red", "purple"]
GRADE_DESC = {
    0: "No signs of diabetic retinopathy detected.",
    1: "Mild non-proliferative DR. Microaneurysms present.",
    2: "Moderate non-proliferative DR. More than just microaneurysms.",
    3: "Severe non-proliferative DR. Significant vascular changes.",
    4: "Proliferative DR. Neovascularization and/or vitreous hemorrhage.",
}

# ——— Model ———
class DRModel(nn.Module):
    def __init__(self, backbone="tf_efficientnetv2_b1", num_classes=5):
        super().__init__()
        self.backbone = timm.create_model(backbone, pretrained=False, num_classes=0)
        in_features = self.backbone.num_features
        self.head = nn.Sequential(
            nn.BatchNorm1d(in_features),
            nn.Linear(in_features, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )
    def forward(self, x):
        return self.head(self.backbone(x))

@st.cache_resource
def load_model():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ckpt = torch.load(MODEL_PATH, map_location=device, weights_only=False)
    model = DRModel(backbone=ckpt.get("backbone", "tf_efficientnetv2_b1"))
    model.load_state_dict(ckpt["model_state"])
    model.to(device).eval()
    with open(THRESH_PATH) as f:
        thresholds = json.load(f)["thresholds"]
    return model, device, thresholds

def preprocess(image_np, size=384):
    img = cv2.cvtColor(image_np, cv2.COLOR_RGB2BGR)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 15, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        cnt = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(cnt)
        img = img[y:y+h, x:x+w]
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    scale = size / max(h, w)
    img = cv2.resize(img, (int(w*scale), int(h*scale)))
    canvas = np.full((size, size, 3), 128, dtype=np.uint8)
    y_off, x_off = (size-img.shape[0])//2, (size-img.shape[1])//2
    canvas[y_off:y_off+img.shape[0], x_off:x_off+img.shape[1]] = img
    blur = cv2.GaussianBlur(canvas.astype(np.float32), (0,0), 10)
    enhanced = np.clip(4.0*canvas - 4.0*blur + 128, 0, 255).astype(np.uint8)
    lab = cv2.cvtColor(enhanced, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    l = clahe.apply(l)
    enhanced = cv2.cvtColor(cv2.merge([l,a,b]), cv2.COLOR_LAB2RGB)
    return enhanced

def predict(model, image, device, thresholds):
    img = preprocess(image)
    tensor = torch.from_numpy(img).permute(2,0,1).float() / 255.0
    mean = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
    std = torch.tensor([0.229,0.224,0.225]).view(3,1,1)
    tensor = (tensor - mean) / std
    tensor = tensor.unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(tensor)
        probs = torch.softmax(output, dim=1).cpu().numpy()[0]
    expected = (probs * np.arange(5)).sum()
    grade = int(np.digitize(expected, thresholds))
    grade = min(grade, 4)
    return grade, probs, img

# ——— Streamlit UI ———
st.set_page_config(page_title="DR Grading", page_icon="🏥", layout="wide")
st.title("🏥 Diabetic Retinopathy Grading System")
st.markdown("Upload a retinal fundus image for automated DR severity grading.")

model, device, thresholds = load_model()

uploaded = st.file_uploader("Upload fundus image", type=["png", "jpg", "jpeg"])

if uploaded:
    image = np.array(Image.open(uploaded).convert("RGB"))
    col1, col2 = st.columns(2)
    with col1:
        st.image(image, caption="Uploaded Image", use_container_width=True)
    grade, probs, processed = predict(model, image, device, thresholds)
    with col2:
        st.image(processed, caption="Preprocessed", use_container_width=True)
    st.markdown(f"### Prediction: Grade {grade} — :{GRADE_COLORS[grade]}[{GRADE_MAP[grade]}]")
    st.info(GRADE_DESC[grade])
    st.bar_chart(dict(zip([GRADE_MAP[i] for i in range(5)], probs.tolist())))
    st.warning("⚠️ This is an AI-assisted tool. Always consult an ophthalmologist.")
'''

app_path = Path("streamlit_app.py")
with open(app_path, 'w') as f:
    f.write(app_code)

print(f"✅ Streamlit app saved: {app_path}")
print(f"\nTo run the app:")
print(f"  streamlit run streamlit_app.py")


## 🤗 Step 29 — Hugging Face Deployment Files
Generates all files needed for Hugging Face Spaces deployment.

In [ ]:
# — Step 29: Hugging Face Deployment —
from pathlib import Path

hf_dir = Path("./huggingface_space")
hf_dir.mkdir(exist_ok=True)

# README.md
readme = '''---
title: Diabetic Retinopathy Grading
emoji: 🏥
colorFrom: blue
colorTo: green
sdk: streamlit
sdk_version: "1.31.0"
app_file: app.py
pinned: false
---

# Diabetic Retinopathy Grading System

Upload a retinal fundus image for automated severity grading using deep learning.

## Model
- **Architecture**: EfficientNetV2-B1
- **Training Data**: APTOS 2019 Blindness Detection
- **Metric**: Quadratic Weighted Kappa (QWK)

## Grades
| Grade | Description |
|-------|-------------|
| 0 | No DR |
| 1 | Mild NPDR |
| 2 | Moderate NPDR |
| 3 | Severe NPDR |
| 4 | Proliferative DR |

⚠️ This is an AI-assisted tool. Always consult an ophthalmologist.
'''

with open(hf_dir / "README.md", 'w') as f:
    f.write(readme)

# requirements.txt
reqs = "torch\ntimm\nopencv-python-headless\nnumpy\nstreamlit\nPillow\n"
with open(hf_dir / "requirements.txt", 'w') as f:
    f.write(reqs)

# Copy app
import shutil
shutil.copy("streamlit_app.py", hf_dir / "app.py")

print(f"✅ Hugging Face Space files created in {hf_dir}/")
print(f"   Files: {[f.name for f in hf_dir.iterdir()]}")
print(f"\nTo deploy:")
print(f"  1. Create a new Space on huggingface.co/spaces")
print(f"  2. Upload the contents of {hf_dir}/")
print(f"  3. Upload model_export/ files to the Space")


## 🏁 Step 30 — Pipeline Summary & Final Report

In [ ]:
# — Step 30: Final Summary —
from pathlib import Path
import json

print("=" * 70)
print("  🏥 DIABETIC RETINOPATHY GRADING PIPELINE — COMPLETE")
print("=" * 70)

print(f"\n📊 RESULTS:")
print(f"   Test QWK (Optimized):  {test_qwk_opt:.4f}")
print(f"   Test QWK (Argmax):     {test_qwk_argmax:.4f}")
print(f"   Test Accuracy:         {test_acc_opt:.4f}")
print(f"   Best Val QWK:          {tracker.best_qwk:.4f}")

print(f"\n🏗️ MODEL:")
print(f"   Backbone:    {BACKBONE}")
print(f"   Resolution:  384×384")
print(f"   Parameters:  {sum(p.numel() for p in model.parameters()):,}")
print(f"   Thresholds:  {[f'{t:.3f}' for t in optimal_thresholds]}")

print(f"\n📁 FILES:")
export_dir = Path("model_export")
if export_dir.exists():
    for f in sorted(export_dir.iterdir()):
        print(f"   {f.name:30s} {f.stat().st_size/1e3:>8.1f} KB")

print(f"\n📦 ARTIFACTS:")
artifact_dir = Path("artifacts")
if artifact_dir.exists():
    for f in sorted(artifact_dir.iterdir()):
        print(f"   {f.name:30s} {f.stat().st_size/1e6:>8.1f} MB")

print(f"\n🚀 DEPLOYMENT:")
print(f"   Streamlit:  streamlit run streamlit_app.py")
print(f"   HuggingFace: Upload ./huggingface_space/ to HF Spaces")

print(f"\n{'='*70}")
print(f"  ✅ All 30 steps completed successfully!")
print(f"{'='*70}")
